## 1. Training Multi-View PAIDF AnomalyGen

You must complete the previous setup step [0-setup-cuda128.ipynb](./0-setup-cuda128.ipynb).

**Important**: Multi-view training is very similar to single-view training. Most configurations are the same, with the main difference being the dataset structure and the addition of `view_types` in the config.

**Note**: For full training, it is recommended to use the command line instead of the notebook, as notebook cells have limited logging capacity.


### 1.0 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.


In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.


### 1.1 Multi-View Dataset Structure

Multi-view datasets have a specific structure where each sample contains multiple views (camera angles) of the same object. The key difference from single-view is:

1. **Images**: Each sample has multiple view images (e.g., `25_bumps_view0.png`, `25_bumps_view1.png`, ...)
2. **Masks**: Can be either **shared** (one mask for all views) or **per-view** (each view has its own mask)

#### Dataset Directory Structure

```
datasets/<DatasetName>/<SampleName>/
├── anomaly_image/
│   └── <AnomalyType>/
│       ├── <sample>_view0.png  # View 0 image
│       ├── <sample>_view1.png  # View 1 image
│       └── ...
└── mask/
    └── <AnomalyType>/
        ├── <sample>_mask.png           # Shared mask (all views use this)
        # OR
        ├── <sample>_view0_mask.png     # Per-view mask for view 0
        ├── <sample>_view1_mask.png    # Per-view mask for view 1
        └── ...
```

#### Example 1: PeppermintCandy (Shared Mask)

The PeppermintCandy dataset uses **shared masks** - one mask file for all views:

- Images: `25_bumps_view0.png`, `25_bumps_view1.png`, ..., `25_bumps_view5.png`
- Mask: `25_bumps_mask.png` (shared by all 6 views)

#### Example 2: SimCardSet (Per-View Mask)

The SimCardSet dataset uses **per-view masks** - each view has its own mask:

- Images: `S0001_view1.jpg`, `S0001_view2.jpg`, ..., `S0001_view5.jpg`
- Masks: `S0001_view1_mask.png`, `S0001_view2_mask.png`, ..., `S0001_view5_mask.png`

**Note**: The dataset loader automatically detects whether masks are shared or per-view based on the file naming pattern.


### 1.2 Training Configuration

Multi-view training configuration is almost identical to single-view, with a few key additions.

#### Key Differences from Single-View:

1. **`view_types`**: List of view names in `dataloader_train.dataset` (e.g., `["view0", "view1", ..., "view5"]`)
2. **`pipe_config`**: Model pipeline configuration in `model.config.pipe_config` - **must match the number of views**
3. **`batch_size`**: Typically set to 1 for multi-view (more memory intensive)

#### Important: `pipe_config` for Multi-View

The `pipe_config` section in `model.config` is **critical** for multi-view training. It must specify:
- **`state_t`**: Must match the number of views (temporal dimension for multi-view)
- **`net.num_views`**: Number of camera views for MultiViewDiT

**These values must match the number of views in `view_types`!**

#### Example Configuration (PeppermintCandy - 6 views):

```yaml
dataloader_train:
  batch_size: 1  # Small batch size for multi-view
  dataset:
    dataset_dir: datasets/PeppermintCandy_multiview
    view_types:
      - view0
      - view1
      - view2
      - view3
      - view4
      - view5
    anomaly_types:
      - 
        - PeppermintCandy
        - bumps
      - 
        - PeppermintCandy
        - colors
      - 
        - PeppermintCandy
        - dents
      - 
        - PeppermintCandy
        - normals

model:
  config:
    pipe_config:
      state_t: 6  # Must match number of views (6 for PeppermintCandy)
      net:
        num_views: 6  # Number of camera views for MultiViewDiT
```

#### Example Configuration (SimCardSet - 5 views):

```yaml
dataloader_train:
  batch_size: 1
  dataset:
    dataset_dir: datasets/SimCardSet
    view_types:
      - view1
      - view2
      - view3
      - view4
      - view5
    anomaly_types:
      - 
        - SimCardSet
        - CH
      - 
        - SimCardSet
        - HS
      - 
        - SimCardSet
        - ZW

model:
  config:
    pipe_config:
      state_t: 5  # Must match number of views (5 for SimCardSet)
      net:
        num_views: 5  # Number of camera views for MultiViewDiT
```

**Note**: The `anomaly_types` format uses a nested list structure (each anomaly type is a list of `[sample_name, anomaly_name]`).


### 1.3 Training Commands

The training command is identical to single-view, just use a multi-view config file.

#### Example 1: PeppermintCandy (6 views, shared masks)


<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.ag_train \
    --config=cosmos_predict2/configs/base/ag_config.py \
    --ag_config=ag_configs/PeppermintCandy_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0.yaml \
    -- experiment=predict2_anomaly_gen_multiview_ddp_2b
```

</details>


In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
        CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.ag_train \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_config=ag_configs/PeppermintCandy_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0.yaml \
        -- experiment=predict2_anomaly_gen_multiview_ddp_2b"


#### Example 2: SimCardSet (5 views, per-view masks)


<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.ag_train \
    --config=cosmos_predict2/configs/base/ag_config.py \
    --ag_config=ag_configs/SimCardSet_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0.yaml \
    -- experiment=predict2_anomaly_gen_multiview_ddp_2b
```

</details>


In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
        CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.ag_train \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_config=ag_configs/SimCardSet_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0.yaml \
        -- experiment=predict2_anomaly_gen_multiview_ddp_2b"


### 1.4 Validation Output

Multi-view validation saves results for **all views**, not just the first one. The output structure is:

```
results/<project>/<group>/<name>/valid/<step>/
├── original_image/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
├── original_mask/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
├── reconstructed_image/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
└── valid_kpi.csv  # Per-view KPI table
```

The `valid_kpi.csv` includes:
- FID scores per view (e.g., `view0`, `view1`, ...)
- Average FID per anomaly type (across all views)
- Overall average FID (across all anomaly types and views)


## Next Step

You can now proceed to the inference step: [3-generation-multiview.ipynb](./3-generation-multiview.ipynb).
